# 🤖 โปรเจกต์ ML: Model Training (การสร้างและสอนโมเดล)
**เป้าหมาย:** สร้างโมเดล Machine Learning แบบ **Ensemble** ที่ประกอบด้วยอัลกอริทึม 3 ประเภท เพื่อทำนายรายได้นักศึกษา (Regression)
1. **Random Forest** (ตัวแทนสาย Bagging Tree - เน้นความเสถียร)
2. **Gradient Boosting** (ตัวแทนสาย Boosting Tree - เน้นความแม่นยำ)
3. **SVR (Support Vector Regressor)** (ตัวแทนสายสมการคณิตศาสตร์ - สร้างความหลากหลายให้โมเดล)

## 1) Import Libraries (นำเข้าเครื่องมือที่จำเป็น)
โหลดไลบรารีพื้นฐานสำหรับการจัดการตารางข้อมูล (Pandas) และการบันทึกไฟล์ (Joblib)

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore') # ปิดแจ้งเตือนสีแดงยิบย่อย

print("✅ Import Libraries เรียบร้อยพร้อมลุย!")

✅ Import Libraries เรียบร้อยพร้อมลุย!


## 2) โหลดข้อมูลและแบ่งชุดข้อมูล (Load & Split Data)
โหลดไฟล์ `student_spending_cleaned.csv` ที่เราทำความสะอาดไว้ จากนั้นแบ่งข้อมูลเป็น 2 ส่วน:
- **Training Set (80%):** ให้ AI ใช้เรียนรู้หารูปแบบ
- **Testing Set (20%):** เก็บไว้เป็น "ข้อสอบ" เพื่อทดสอบความแม่นยำของ AI

In [2]:
# 1. โหลดข้อมูลที่ผ่านการ Clean มาแล้ว
df = pd.read_csv('../data/student_spending_cleaned.csv')

# 2. แยกตัวแปรต้น (Features: X) และตัวแปรตาม (Target: y)
# เป้าหมายคือการทำนาย monthly_income จึงต้องแยกมันออกมาเป็น y
X = df.drop(columns=['monthly_income'])
y = df['monthly_income']

# 3. แบ่งชุดข้อมูล Train 80% / Test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 จำนวนข้อมูลสำหรับสอนโมเดล (Train): {X_train.shape[0]} แถว")
print(f"📊 จำนวนข้อมูลสำหรับทดสอบ (Test): {X_test.shape[0]} แถว")

📊 จำนวนข้อมูลสำหรับสอนโมเดล (Train): 800 แถว
📊 จำนวนข้อมูลสำหรับทดสอบ (Test): 200 แถว


## 3) การปรับสเกลข้อมูล (Feature Scaling)
อัลกอริทึมอย่าง SVR อ่อนไหวต่อสเกลของตัวเลขที่ต่างกันมาก (เช่น ค่าเทอมหลักหมื่น vs ค่าความบันเทิงหลักร้อย) 
เราจึงต้องใช้ `StandardScaler` เพื่อบีบตัวเลขทุกคอลัมน์ให้อยู่ในมาตรฐานเดียวกัน (Mean=0, Std=1) 
และ **ต้องบันทึก Scaler นี้เก็บไว้ใช้แปลงข้อมูลที่ผู้ใช้กรอกเข้ามาจากหน้าเว็บด้วย**

In [3]:
# 1. สร้างตัวปรับสเกล
scaler = StandardScaler()

# 2. เรียนรู้สเกลและปรับค่าเฉพาะชุด Train (ห้ามไปเรียนรู้จากชุด Test เด็ดขาด เพื่อกันข้อสอบรั่ว)
X_train_scaled = scaler.fit_transform(X_train)

# 3. นำไม้บรรทัดที่ได้มาปรับสเกลชุด Test
X_test_scaled = scaler.transform(X_test)

# 4. บันทึก Scaler เก็บไว้ในโฟลเดอร์ models
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/ml_scaler.pkl')

print("💾 บันทึก 'ml_scaler.pkl' เรียบร้อย (สำคัญมากสำหรับหน้า Web App)")

💾 บันทึก 'ml_scaler.pkl' เรียบร้อย (สำคัญมากสำหรับหน้า Web App)


## 4) สร้างและเทรนโมเดล Ensemble (Build Ensemble Model)
เราจะกำหนดโมเดล 3 ประเภท แล้วนำมารวมพลังกันด้วย `VotingRegressor` (หลักการคือให้ทั้ง 3 ตัวช่วยกันทำนาย แล้วเอาผลลัพธ์มาเฉลี่ยกัน เพื่อให้ได้คำตอบที่แม่นยำและเสถียรที่สุด)

In [4]:
# 1. กำหนดโมเดลย่อยทั้ง 3 ตัว (เปลี่ยนจาก XGBoost เป็น GradientBoostingRegressor)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
svr_model = SVR(kernel='linear', C=10) 

# 2. รวมร่างเป็น Ensemble ด้วย VotingRegressor
ensemble_model = VotingRegressor(estimators=[
    ('RandomForest', rf_model),
    ('GradientBoosting', gb_model),
    ('SVR', svr_model)
])

# 3. สั่งเทรนโมเดล (ใช้ข้อมูลที่ Scale แล้ว)
print("🧠 AI กำลังเรียนรู้และสร้างโมเดล Ensemble (รอสักครู่)...")
ensemble_model.fit(X_train_scaled, y_train)

print("✅ เทรนโมเดลเสร็จสมบูรณ์!")

🧠 AI กำลังเรียนรู้และสร้างโมเดล Ensemble (รอสักครู่)...
✅ เทรนโมเดลเสร็จสมบูรณ์!


## 5) ประเมินผลและบันทึกโมเดล (Evaluation & Export)
ทดสอบทำนายผลกับ Testing Set (ข้อสอบ) แล้ววัดผลด้วยค่าสถิติ:
- **$R^2$ (R-squared):** สัดส่วนความแม่นยำ ยิ่งเข้าใกล้ 1 (หรือ 100%) ยิ่งดี
- **MAE (Mean Absolute Error):** ค่าเฉลี่ยความผิดพลาด (หน่วยเป็นดอลลาร์)
- **RMSE:** วัดความผิดพลาดโดยให้ความสำคัญกับค่าที่ทายผิดไปมากๆ

In [5]:
# 1. ให้โมเดลลองทำนายข้อสอบ (Test Set)
y_pred = ensemble_model.predict(X_test_scaled)

# 2. คำนวณความแม่นยำ
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("🎯 === ผลการทดสอบความแม่นยำของโมเดล (Test Set) ===")
print(f"R-squared (R²): {r2:.4f} (อธิบายความผันผวนของรายได้ได้ {r2*100:.2f}%)")
print(f"MAE (คลาดเคลื่อนเฉลี่ย): ±${mae:,.2f}")
print(f"RMSE (คลาดเคลื่อนแบบถ่วงน้ำหนัก): ±${rmse:,.2f}")
print("===================================================\n")

# 3. บันทึกตัวโมเดลและผลลัพธ์เพื่อนำไปโชว์ใน Streamlit
joblib.dump(ensemble_model, '../models/ensemble_model.pkl')
joblib.dump({"mae": mae, "rmse": rmse, "r2": r2}, '../models/ml_metrics.pkl')

print("💾 บันทึก 'ensemble_model.pkl' และ 'ml_metrics.pkl' เรียบร้อย!")
print("🎉 จบกระบวนการฝั่ง Machine Learning! พร้อมเอาไปขึ้นหน้าเว็บแล้วครับ")

🎯 === ผลการทดสอบความแม่นยำของโมเดล (Test Set) ===
R-squared (R²): -0.0102 (อธิบายความผันผวนของรายได้ได้ -1.02%)
MAE (คลาดเคลื่อนเฉลี่ย): ±$247.28
RMSE (คลาดเคลื่อนแบบถ่วงน้ำหนัก): ±$293.08

💾 บันทึก 'ensemble_model.pkl' และ 'ml_metrics.pkl' เรียบร้อย!
🎉 จบกระบวนการฝั่ง Machine Learning! พร้อมเอาไปขึ้นหน้าเว็บแล้วครับ
